In [ ]:
!pip install "jedi>=0.16"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.9 MB/s eta 0:00:00


In [ ]:
!pip -q install -U vllm datasets transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7

In [ ]:
import re
import math
import json
from typing import List, Dict, Any

from datasets import load_dataset
from vllm import LLM, SamplingParams

In [ ]:
MODEL_ID = "allenai/OLMo-2-1124-7B-SFT"
LORA_ADAPTER_PATH = "adapter_config.json"
SPLIT = "test"
MAX_EXAMPLES = 200
K_LIST = [1, 4, 8, 16, 32, 64, 128]
N_SAMPLES = max(K_LIST)
TEMPERATURE = 0.6
TOP_P = 0.95
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16
TENSOR_PARALLEL_SIZE = 1

In [ ]:
import os
import sys

# Patch the ipykernel file on disk so spawned worker processes inherit the fix (Gemini fix)
os.system("sed -i 's/raise io.UnsupportedOperation(\"fileno\")/return 1/g' /usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py")

# In-memory patch for the current process
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.90,
    enable_lora=True,
    max_lora_rank=16,
)

sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
)

INFO 05-03 23:05:40 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'enable_lora': True, 'model': 'allenai/OLMo-2-1124-7B-SFT'}


config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

INFO 05-03 23:05:59 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-03 23:05:59 [nixl_utils.py:34] NIXL is not available
WARNING 05-03 23:05:59 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-03 23:05:59 [model.py:555] Resolved architecture: Olmo2ForCausalLM
INFO 05-03 23:05:59 [model.py:1680] Using max model len 2048
INFO 05-03 23:05:59 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-03 23:05:59 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-03 23:05:59 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

(EngineCore pid=8765) INFO 05-03 23:06:01 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='allenai/OLMo-2-1124-7B-SFT', speculative_config=None, tokenizer='allenai/OLMo-2-1124-7B-SFT', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_trac

Loading pt checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=8765) INFO 05-03 23:06:55 [default_loader.py:384] Loading weights took 12.35 seconds
(EngineCore pid=8765) INFO 05-03 23:06:55 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=8765) INFO 05-03 23:06:56 [gpu_model_runner.py:4879] Model loading took 13.67 GiB memory and 52.200164 seconds
(EngineCore pid=8765) INFO 05-03 23:07:12 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/cfec7e5a28/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=8765) INFO 05-03 23:07:12 [backends.py:1128] Dynamo bytecode transform time: 15.58 s
(EngineCore pid=8765) INFO 05-03 23:07:19 [backends.py:376] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=8765) INFO 05-03 23:07:29 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 16.58 s
(EngineCore pid=8765) INFO 05-03 23:07:37 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a2dad4cf0caf59130a6

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:19<00:00,  5.13it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 70/70 [00:07<00:00,  9.86it/s]


(EngineCore pid=8765) INFO 05-03 23:08:23 [gpu_model_runner.py:6133] Graph capturing finished in 28 secs, took 1.11 GiB
(EngineCore pid=8765) INFO 05-03 23:08:23 [gpu_worker.py:599] CUDA graph pool memory: 1.11 GiB (actual), 0.95 GiB (estimated), difference: 0.16 GiB (14.4%).
(EngineCore pid=8765) INFO 05-03 23:08:23 [core.py:299] init engine (profile, create kv cache, warmup model) took 87.47 s (compilation: 40.50 s)
(EngineCore pid=8765) INFO 05-03 23:08:24 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


In [ ]:
def build_prompt(question):
    return (
        "<|im_start|>system\n"
        "You are a helpful assistant.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{question}\n"
        "Please reason step by step, and put your final answer within\\boxed{{}}.<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

In [ ]:
_gold_re = re.compile(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)")

def normalize_number_str(s):
    s = s.strip().replace(",", "")
    if re.fullmatch(r"[-+]?\d+\.0+", s):
        s = s.split(".")[0]
    return s

def extract_gold_answer(answer_text):
    m = _gold_re.search(answer_text)
    if m:
        return normalize_number_str(m.group(1))
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", answer_text)
    return normalize_number_str(nums[-1]) if nums else ""

def extract_pred_answer(completion):
    m = re.search(r"\\boxed\{([^}]*)\}", completion)
    if m:
        inside = m.group(1)
        nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", inside)
        if nums:
            return normalize_number_str(nums[-1])

    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", completion)
    if nums:
        return normalize_number_str(nums[-1])

    return ""

In [ ]:
def estimate_pass_at_k(n: int, c: int, k: int) -> float:
    """
    Unbiased estimator:
    pass@k = 1 - C(n-c, k) / C(n, k)
    """
    if c <= 0:
        return 0.0
    if n - c < k:
        return 1.0
    def log_comb(a: int, b: int) -> float:
        return math.lgamma(a + 1) - math.lgamma(b + 1) - math.lgamma(a - b + 1)

    log_num = log_comb(n - c, k)
    log_den = log_comb(n, k)
    ratio = math.exp(log_num - log_den)
    return 1.0 - ratio

In [ ]:
ds = load_dataset("HuggingFaceH4/aime_2024", split="train")
#ds = load_dataset("gsm8k", "main", split="test")

if MAX_EXAMPLES is not None:
    ds = ds.select(range(min(MAX_EXAMPLES, len(ds))))

examples = []
for ex in ds:
    examples.append({
        "question": ex["problem"], #"question": ex["question"],
        "gold": extract_gold_answer(ex["answer"]), #"gold": extract_gold_answer(ex["answer"]),
        "prompt": build_prompt(ex["problem"]), #"prompt": build_prompt(ex["question"]),
    })

print(f"Loaded {len(examples)} AIME 2024 examples.")


README.md:   0%|          | 0.00/932 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/81.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

Loaded 30 AIME 2024 examples.


In [ ]:
import os
from vllm.lora.request import LoRARequest

all_results: List[Dict[str, Any]] = []

# LoRARequest requires a directory path, not a file path.
if LORA_ADAPTER_PATH:
    adapter_dir = os.path.dirname(LORA_ADAPTER_PATH)
    # If LORA_ADAPTER_PATH was just the file name, it means the current directory
    if not adapter_dir:
        adapter_dir = "."
    lora_req = LoRARequest("custom_adapter", 1, adapter_dir)
else:
    lora_req = None

for start in range(0, len(examples), BATCH_SIZE):
    batch = examples[start:start + BATCH_SIZE]
    prompts = [x["prompt"] for x in batch]

    outputs = llm.generate(prompts, sampling_params, lora_request=lora_req)

    for ex, out in zip(batch, outputs):
        preds = []
        correct_flags = []

        for cand in out.outputs:
            text = cand.text
            pred = extract_pred_answer(text)
            is_correct = (pred != "" and pred == ex["gold"])

            preds.append({
                "text": text,
                "pred": pred,
                "correct": is_correct,
            })
            correct_flags.append(is_correct)

        c = sum(correct_flags)

        row = {
            "question": ex["question"],
            "gold": ex["gold"],
            "num_correct": c,
            "samples": preds,
        }

        for k in K_LIST:
            row[f"pass@{k}"] = estimate_pass_at_k(N_SAMPLES, c, k)

        all_results.append(row)

    print(f"Processed {min(start + BATCH_SIZE, len(examples))}/{len(examples)}")

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

WARNING 05-03 23:08:41 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/2048 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 16/30


Rendering prompts:   0%|          | 0/14 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1792 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 30/30


In [ ]:
metrics = {}
for k in K_LIST:
    metrics[f"pass@{k}"] = sum(r[f"pass@{k}"] for r in all_results) / len(all_results)

print("\n=== Final Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


with open("math500_passk_results.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": MODEL_ID,
            "num_examples": len(all_results),
            "n_samples": N_SAMPLES,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "metrics": metrics,
            "results": all_results,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved detailed results to lora_aime2024_passk_results.json")


=== Final Metrics ===
pass@1: 0.0003
pass@4: 0.0010
pass@8: 0.0021
pass@16: 0.0042
pass@32: 0.0083
pass@64: 0.0167
pass@128: 0.0333

Saved detailed results to lora_aime2024_passk_results.json
